In [ ]:
import pandas as pd
from google.colab import files

# Step 1: Upload the dataset
print("Please upload your dataset.")
uploaded = files.upload()

# Assuming the uploaded file is named 'power consumption.csv'
file_name = list(uploaded.keys())[0]
data = pd.read_csv(file_name)

# Step 2: Parse the DateTime column as datetime and set it as the index
# Let pandas infer the date format automatically
data['DateTime'] = pd.to_datetime(data['DateTime'], infer_datetime_format=True, errors='coerce')

# Step 3: Drop rows with invalid DateTime values
data.dropna(subset=['DateTime'], inplace=True)

# Step 4: Set DateTime as the index and sort the data
data.set_index('DateTime', inplace=True)
data.sort_index(inplace=True)

# Step 5: Check for missing values
missing_values = data.isnull().sum()

# Print summary of data preparation
print("Dataset Info:")
print(data.info())
print("\nMissing Values:")
print(missing_values)

# Display the first few rows
data.head()


Visualize Time-Series Trends

In [ ]:
# Step 1: Clean column names by stripping extra spaces
data.columns = data.columns.str.strip()

# Step 2: Re-plot power consumption trends with enhanced formatting
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 10), dpi=600)

# Plot Zone 1
plt.plot(data.index, data['Zone 1'], label='Zone 1', alpha=0.7, linewidth=2)
# Plot Zone 2
plt.plot(data.index, data['Zone 2'], label='Zone 2', alpha=0.7, linewidth=2)
# Plot Zone 3
plt.plot(data.index, data['Zone 3'], label='Zone 3', alpha=0.7, linewidth=2)

# Add labels, legend, and title with bold formatting
plt.title('Power Consumption Trends (Zones 1, 2, and 3)', fontsize=22, fontweight='bold')
plt.xlabel('DateTime', fontsize=20, fontweight='bold')
plt.ylabel('Power Consumption', fontsize=20, fontweight='bold')

# Bold axis numbers and increase font size
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')

# Enhance legend formatting
plt.legend(fontsize=14, loc='upper right', frameon=True)

plt.grid(alpha=0.4)
plt.tight_layout()

# Show the plot
plt.show()


Code for Decomposition:

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Decompose the time series for Zone 1
decomposition = seasonal_decompose(data['Zone 1'], model='additive', period=144)  # Assuming 144 points/day (10-min intervals)

# Plot the decomposed components
plt.figure(figsize=(14, 12), dpi=600)

# Trend
plt.subplot(4, 1, 1)
plt.plot(decomposition.trend, label='Trend', color='blue')
plt.title('Trend Component (Zone 1)', fontsize=16, fontweight='bold')
plt.ylabel('Power Consumption (Units)', fontsize=14, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.grid(alpha=0.4)

# Seasonality
plt.subplot(4, 1, 2)
plt.plot(decomposition.seasonal, label='Seasonality', color='green')
plt.title('Seasonal Component (Zone 1)', fontsize=16, fontweight='bold')
plt.ylabel('Power Consumption (Units)', fontsize=14, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.grid(alpha=0.4)

# Residuals
plt.subplot(4, 1, 3)
plt.plot(decomposition.resid, label='Residuals', color='red')
plt.title('Residual Component (Zone 1)', fontsize=16, fontweight='bold')
plt.ylabel('Residuals', fontsize=14, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.grid(alpha=0.4)

# Observed
plt.subplot(4, 1, 4)
plt.plot(data['Zone 1'], label='Observed', color='black')
plt.title('Observed Data (Zone 1)', fontsize=16, fontweight='bold')
plt.xlabel('DateTime', fontsize=14, fontweight='bold')
plt.ylabel('Power Consumption (Units)', fontsize=14, fontweight='bold')
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.grid(alpha=0.4)

plt.tight_layout()
plt.show()


Perform the Augmented Dickey-Fuller (ADF) Test

In [ ]:
# Perform ADF test for all zones
zones = ['Zone 1', 'Zone 2', 'Zone 3']
for zone in zones:
    result = adfuller(data[zone].dropna())
    print(f"\nADF Test Results for {zone}:")
    print(f"Test Statistic: {result[0]:.4f}")
    print(f"P-Value: {result[1]:.4f}")
    print("Critical Values:")
    for key, value in result[4].items():
        print(f"   {key}: {value:.4f}")

    # Interpretation of Results
    if result[1] <= 0.05:
        print(f"The series for {zone} is stationary (p-value <= 0.05).")
    else:
        print(f"The series for {zone} is non-stationary (p-value > 0.05).")


ACF and PACF Plots

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt

# Define zones for analysis
zones = ['Zone 1', 'Zone 2', 'Zone 3']

# Plot ACF and PACF for each zone
for zone in zones:
    plt.figure(figsize=(14, 6), dpi=600)

    # ACF Plot
    plt.subplot(1, 2, 1)
    plot_acf(data[zone].dropna(), lags=50, ax=plt.gca())
    plt.title(f'Autocorrelation Function (ACF) - {zone}', fontsize=16, fontweight='bold')
    plt.xlabel('Lags', fontsize=14, fontweight='bold')
    plt.ylabel('ACF', fontsize=14, fontweight='bold')
    plt.xticks(fontsize=12, fontweight='bold')
    plt.yticks(fontsize=12, fontweight='bold')

    # PACF Plot
    plt.subplot(1, 2, 2)
    plot_pacf(data[zone].dropna(), lags=50, ax=plt.gca(), method='ywm')
    plt.title(f'Partial Autocorrelation Function (PACF) - {zone}', fontsize=16, fontweight='bold')
    plt.xlabel('Lags', fontsize=14, fontweight='bold')
    plt.ylabel('PACF', fontsize=14, fontweight='bold')
    plt.xticks(fontsize=12, fontweight='bold')
    plt.yticks(fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.show()



ARIMA MODEL

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
import matplotlib.pyplot as plt
import os

# Make sure your save folder exists
save_path = r"C:\MyPlots"
os.makedirs(save_path, exist_ok=True)

# Fit ARIMA model
arima_model = ARIMA(data['Zone 1'], order=(1, 0, 1))
arima_result = arima_model.fit()

# --- Plot Actual vs Fitted ---
plt.figure(figsize=(14, 10), dpi=1800)
plt.plot(data['Zone 1'], label='Observed', color='blue', alpha=0.7)
plt.plot(arima_result.fittedvalues, label='Fitted', color='red', alpha=0.7)
plt.title('ARIMA Model - Zone 1: Observed vs Fitted', fontsize=24, fontweight='bold')
plt.xlabel('Date', fontsize=16, fontweight='bold')
plt.ylabel('Power Consumption', fontsize=16, fontweight='bold')
plt.legend(fontsize=30, loc='best', frameon=True)
plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')
plt.grid(alpha=0.4)
plt.tight_layout()

# Save before plt.show()
plt.savefig(os.path.join(save_path, "ARIMA_Observed_vs_Fitted.tiff"), dpi=100, format='tiff')
plt.show()

# --- Forecast next 50 steps ---
forecast = arima_result.forecast(steps=50)

# --- Plot Forecast ---
plt.figure(figsize=(14, 10), dpi=1800)
plt.plot(data['Zone 1'], label='Observed', color='blue', alpha=0.7)
plt.plot(forecast, label='Forecast', color='green', alpha=1.0)
plt.title('ARIMA Model - Zone 1: Forecast', fontsize=24, fontweight='bold')
plt.xlabel('Date', fontsize=16, fontweight='bold')
plt.ylabel('Power Consumption', fontsize=16, fontweight='bold')
plt.legend(fontsize=30, loc='best', frameon=True)
plt.xticks(fontsize=16, fontweight='bold')
plt.yticks(fontsize=16, fontweight='bold')
plt.grid(alpha=0.4)
plt.tight_layout()

# Save before plt.show()
plt.savefig(os.path.join(save_path, "ARIMA_Forecast.tiff"), dpi=100, format='tiff')
plt.show()


Code to Fit ARIMA Model:

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
import matplotlib.pyplot as plt

# Fit ARIMA model
arima_model = ARIMA(data['Zone 1'], order=(1, 0, 1))  # p=1, d=0, q=1
arima_result = arima_model.fit()

# Print model summary
print(arima_result.summary())

# Plot actual vs. fitted values
plt.figure(figsize=(14, 8), dpi=1800)
plt.plot(data['Zone 1'], label='Observed', color='blue', alpha=0.7)
plt.plot(arima_result.fittedvalues, label='Fitted', color='red', alpha=0.7)
plt.title('ARIMA Model - Zone 1: Observed vs Fitted', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=14, fontweight='bold')
plt.ylabel('Power Consumption', fontsize=14, fontweight='bold')
plt.legend(fontsize=14)
plt.xticks(fontsize=12, fontweight='bold')
plt.yticks(fontsize=12, fontweight='bold')
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

# Forecast next 50 steps
forecast = arima_result.forecast(steps=50)

# Plot forecast
plt.figure(figsize=(14, 8), dpi=1800)
plt.plot(data['Zone 1'], label='Observed', color='blue', alpha=0.7)
plt.plot(forecast, label='Forecast', color='green', alpha=0.7)
plt.title('ARIMA Model - Zone 1: Forecast', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=14, fontweight='bold')
plt.ylabel('Power Consumption', fontsize=14, fontweight='bold')
plt.legend(fontsize=14)
plt.xticks(fontsize=12, fontweight='bold')
plt.yticks(fontsize=12, fontweight='bold')
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
import matplotlib.pyplot as plt
import os

# Directory to save plots
save_dir = os.path.join(os.getcwd(), "saved_plots")
os.makedirs(save_dir, exist_ok=True)

# Fit ARIMA model
arima_model = ARIMA(data['Zone 1'], order=(1, 0, 1))  # p=1, d=0, q=1
arima_result = arima_model.fit()

# Print model summary
print(arima_result.summary())

# ---------- Plot 1: Observed vs Fitted ----------
plt.figure(figsize=(14, 8), dpi=100)
plt.plot(data['Zone 1'], label='Observed', color='blue', alpha=0.7)
plt.plot(arima_result.fittedvalues, label='Fitted', color='red', alpha=0.7)
plt.title('ARIMA Model - Zone 1: Observed vs Fitted', fontsize=18, fontweight='bold')
plt.xlabel('Date', fontsize=16, fontweight='bold')
plt.ylabel('Power Consumption', fontsize=16, fontweight='bold')
plt.legend(fontsize=16, frameon=True, loc='best')  # Bigger legend font
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.grid(alpha=0.4)
plt.tight_layout()

# Save and show
plot1_path = os.path.join(save_dir, "ARIMA_Observed_vs_Fitted.tiff")
plt.savefig(plot1_path, dpi=100, format='tiff')
plt.show()

# ---------- Forecast ----------
forecast = arima_result.forecast(steps=50)

plt.figure(figsize=(14, 8), dpi=100)
plt.plot(data['Zone 1'], label='Observed', color='blue', alpha=0.7)
plt.plot(forecast, label='Forecast', color='green', alpha=0.7)
plt.title('ARIMA Model - Zone 1: Forecast', fontsize=18, fontweight='bold')
plt.xlabel('Date', fontsize=16, fontweight='bold')
plt.ylabel('Power Consumption', fontsize=16, fontweight='bold')
plt.legend(fontsize=16, frameon=True, loc='best')  # Bigger legend font
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.grid(alpha=0.4)
plt.tight_layout()

# Save and show
plot2_path = os.path.join(save_dir, "ARIMA_Forecast.tiff")
plt.savefig(plot2_path, dpi=100, format='tiff')
plt.show()

print(f"✅ Plots saved in: {save_dir}")
save_path = r"C:\Users\YourName\Documents"
plt.savefig(f"{save_path}\\arima_fitted_plot.tiff", format='tiff', dpi=100)


In [ ]:
from statsmodels.tsa.arima.model import ARIMA
import matplotlib.pyplot as plt
import os

# Directory to save plots on your local machine
save_path = r"C:\Users\YourName\Documents"  # Change to your folder path
os.makedirs(save_path, exist_ok=True)

# Fit ARIMA model
arima_model = ARIMA(data['Zone 1'], order=(1, 0, 1))  # p=1, d=0, q=1
arima_result = arima_model.fit()

# Print model summary
print(arima_result.summary())

# ---------- Plot 1: Observed vs Fitted ----------
plt.figure(figsize=(14, 8), dpi=300)
plt.plot(data['Zone 1'], label='Observed', color='blue', alpha=0.7)
plt.plot(arima_result.fittedvalues, label='Fitted', color='red', alpha=0.7)
plt.title('ARIMA Model - Zone 1: Observed vs Fitted', fontsize=18, fontweight='bold')
plt.xlabel('Date', fontsize=16, fontweight='bold')
plt.ylabel('Power Consumption', fontsize=16, fontweight='bold')

# Bold legend
legend = plt.legend(fontsize=16, frameon=True, loc='best')
for text in legend.get_texts():
    text.set_fontweight('bold')

plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.grid(alpha=0.4)
plt.tight_layout()

# Save as TIFF
plot1_file = os.path.join(save_path, "ARIMA_Observed_vs_Fitted.tiff")
plt.savefig(plot1_file, dpi=300, format='tiff')
plt.show()

# ---------- Forecast ----------
forecast = arima_result.forecast(steps=50)

plt.figure(figsize=(14, 8), dpi=300)
plt.plot(data['Zone 1'], label='Observed', color='blue', alpha=0.7)
plt.plot(forecast, label='Forecast', color='green', alpha=0.7)
plt.title('ARIMA Model - Zone 1: Forecast', fontsize=18, fontweight='bold')
plt.xlabel('Date', fontsize=16, fontweight='bold')
plt.ylabel('Power Consumption', fontsize=16, fontweight='bold')

# Bold legend
legend = plt.legend(fontsize=16, frameon=True, loc='best')
for text in legend.get_texts():
    text.set_fontweight('bold')

plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')
plt.grid(alpha=0.4)
plt.tight_layout()

# Save as TIFF
plot2_file = os.path.join(save_path, "ARIMA_Forecast.tiff")
plt.savefig(plot2_file, dpi=300, format='tiff')
plt.show()

print(f"✅ Plots saved in: {save_path}")


In [ ]:
# Step 1: Strip extra spaces from column names
data.columns = data.columns.str.strip()

# Step 2: Prepare the dataset for VAR
var_data = data[['Zone 1', 'Zone 2', 'Zone 3']].copy()
var_data.dropna(inplace=True)  # Drop missing values if any

from statsmodels.tsa.api import VAR
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Step 2: Split data into training and testing sets
train_size = int(len(var_data) * 0.8)
train_data = var_data.iloc[:train_size]
test_data = var_data.iloc[train_size:]

# Step 3: Fit VAR model
var_model = VAR(train_data)
var_result = var_model.fit(maxlags=5)

print(var_result.summary())

# Step 4: Forecast
forecast_steps = len(test_data)
forecast = var_result.forecast(y=train_data.values, steps=forecast_steps)

# Convert forecast into a DataFrame
forecast_df = pd.DataFrame(forecast, index=test_data.index, columns=var_data.columns)

# Step 5: Evaluate model performance
rmse_zone1 = np.sqrt(mean_squared_error(test_data['Zone 1'], forecast_df['Zone 1']))
mae_zone1 = mean_absolute_error(test_data['Zone 1'], forecast_df['Zone 1'])

rmse_zone2 = np.sqrt(mean_squared_error(test_data['Zone 2'], forecast_df['Zone 2']))
mae_zone2 = mean_absolute_error(test_data['Zone 2'], forecast_df['Zone 2'])

rmse_zone3 = np.sqrt(mean_squared_error(test_data['Zone 3'], forecast_df['Zone 3']))
mae_zone3 = mean_absolute_error(test_data['Zone 3'], forecast_df['Zone 3'])

print(f"Zone 1 - RMSE: {rmse_zone1:.2f}, MAE: {mae_zone1:.2f}")
print(f"Zone 2 - RMSE: {rmse_zone2:.2f}, MAE: {mae_zone2:.2f}")
print(f"Zone 3 - RMSE: {rmse_zone3:.2f}, MAE: {mae_zone3:.2f}")

# Step 6: Plot observed vs predicted for each zone
for zone in var_data.columns:
    plt.figure(figsize=(14, 8), dpi=1800)
    plt.plot(test_data[zone], label='Observed', color='blue', alpha=0.7)
    plt.plot(forecast_df[zone], label='Forecast', color='green', alpha=1)
    plt.title(f'VAR Model - {zone}: Observed vs Forecast', fontsize=22, fontweight='bold')
    plt.xlabel('Date', fontsize=16, fontweight='bold')  # Changed from DateTime → Date
    plt.ylabel('Power Consumption', fontsize=16, fontweight='bold')
    plt.legend(fontsize=22, frameon=True, prop={'weight': 'bold'})  # Bold + bigger font
    plt.xticks(fontsize=14, fontweight='bold')
    plt.yticks(fontsize=14, fontweight='bold')
    plt.grid(alpha=0.4)
    plt.tight_layout()
    plt.show()
